# 从零复现经典 CNN：LeNet、AlexNet、VGG 与 ResNet

这份 notebook 不调用 `torchvision.models`，而是用 PyTorch 基础算子逐层实现四条经典架构脉络：`LeNet`、缩小版 `AlexNet`、由 `VGGBlock` 组成的 `VGGMini`，以及带投影捷径的 `MiniResNet`。每个模型都真正定义 `class` 与 `forward`，并通过 shape hook、参数量、梯度、微型过拟合和状态指纹验证。

数据是离线生成的 32×32 几何图案，目的是验证实现和训练链路，不代表 ImageNet 或任何真实视觉任务的精度、吞吐和鲁棒性。


## 1. 复现边界与验收路线

```text
卷积尺寸/感受野公式
  -> LeNet：卷积 + 全连接
  -> AlexNetMini：更深卷积 + ReLU/Dropout
  -> VGG：重复 3×3 block
  -> ResNet：残差恒等/投影路径
  -> 初始化、hook、参数量
  -> 合成小样本过拟合
  -> BN train/eval、梯度、manifest
```

“从零复现”在这里指不导入现成网络架构；`Conv2d`、`BatchNorm2d`、`Linear`、优化器等张量基础组件仍由 PyTorch 提供。完整复刻论文还需要原始数据、增强、训练日程、正则化与大规模算力，本 notebook 不声称完成这些外部条件。


In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from collections import OrderedDict  # 导入本单元所需的依赖。
from copy import deepcopy  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。


## 2. SAME、VALID、stride 与感受野

对单个空间维度，卷积输出长度为

$$
L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor.
$$

- `VALID` 表示不补零，即 $p=0$；尺寸通常缩小。
- 常见的奇数核、`stride=1` 的 `SAME` 可取 $p=(k-1)/2$，保持尺寸；当 stride 大于 1 时，框架的 SAME 语义需要明确上下两侧可能不对称。
- stride 不只是下采样：它同时扩大后续单元在原图上的“跳距”。逐层递推有效感受野：$j_l=j_{l-1}s_l$，$r_l=r_{l-1}+(k_l-1)d_lj_{l-1}$。


In [ ]:
def conv_out(size, kernel, stride=1, padding=0, dilation=1):  # 定义本节可复用的核心函数。
    numerator = size + 2 * padding - dilation * (kernel - 1) - 1  # 计算并保存当前步骤的中间状态。
    if numerator < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("卷积核的有效尺寸大于补零后的输入")  # 遇到非法合同立即显式失败。
    return numerator // stride + 1  # 返回当前分支计算出的结果。

def receptive_field(layers):  # 定义本节可复用的核心函数。
    jump, field = 1, 1  # 计算并保存当前步骤的中间状态。
    trace = []  # 计算并保存当前步骤的中间状态。
    for kernel, stride, dilation in layers:  # 遍历输入元素以累积或检查结果。
        field = field + (kernel - 1) * dilation * jump  # 计算并保存当前步骤的中间状态。
        jump = jump * stride  # 计算并保存当前步骤的中间状态。
        trace.append({"kernel": kernel, "stride": stride, "jump": jump, "field": field})  # 执行当前语句以推进本节示例。
    return trace  # 返回当前分支计算出的结果。

assert conv_out(32, 3, padding=1) == 32       # 奇数核 stride=1 的 SAME
assert conv_out(32, 3, padding=0) == 30       # VALID；中文说明：该行遵循既定约束。
assert conv_out(31, 3, stride=2, padding=1) == 16  # 用受控断言验证关键不变量。
rf_trace = receptive_field([(3, 1, 1), (2, 2, 1), (3, 1, 1)])  # 计算并保存当前步骤的中间状态。
assert rf_trace[-1]["field"] == 8  # 用受控断言验证关键不变量。
assert rf_trace[-1]["jump"] == 2  # 用受控断言验证关键不变量。
print(rf_trace)  # 执行当前语句以推进本节示例。


## 3. LeNet：局部连接与参数共享

LeNet-5 建立了“卷积提取局部模式—下采样—全连接分类”的基本范式。下面保持两层卷积和三层分类头，但使用现代常见的 ReLU 与最大池化，并将类别数参数化。因此它是教学复现，不是逐位对齐原论文的激活函数、平均池化与连接表。

32×32 输入经过 `5×5 VALID -> 2×2 pool -> 5×5 VALID -> 2×2 pool` 后得到 16×5×5，分类头的 400 维输入是由公式推导，不靠试错猜测。


In [ ]:
class LeNet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_channels <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("in_channels 必须为正")  # 遇到非法合同立即显式失败。
        self.in_channels = in_channels  # 计算并保存当前步骤的中间状态。
        self.features = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(in_channels, 6, kernel_size=5),  # 计算并保存当前步骤的中间状态。
            nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
            nn.MaxPool2d(kernel_size=2, stride=2),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(6, 16, kernel_size=5),  # 计算并保存当前步骤的中间状态。
            nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
            nn.MaxPool2d(kernel_size=2, stride=2),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        self.classifier = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Flatten(), nn.Linear(16 * 5 * 5, 120), nn.ReLU(),  # 执行当前语句以推进本节示例。
            nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, num_classes),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels or x.shape[-2:] != (32, 32):  # 按当前条件选择后续控制路径。
            raise ValueError(f"LeNet 期望 [N,{self.in_channels},32,32]，收到 {tuple(x.shape)}")  # 遇到非法合同立即显式失败。
        return self.classifier(self.features(x))  # 返回当前分支计算出的结果。

lenet = LeNet().to(DEVICE)  # 计算并保存当前步骤的中间状态。
lenet_logits = lenet(torch.zeros(2, 1, 32, 32))  # 计算并保存当前步骤的中间状态。
lenet_rgb = LeNet(in_channels=3).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    lenet_rgb_logits = lenet_rgb(torch.zeros(2, 3, 32, 32))  # 计算并保存当前步骤的中间状态。
assert lenet_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert lenet_rgb_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert lenet_rgb.features[0].in_channels == 3  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in lenet.parameters()) > 50_000  # 用受控断言验证关键不变量。


## 4. AlexNetMini：更深、更宽与 Dropout

AlexNet 的历史意义还包括 ReLU、大规模 GPU 训练、重叠池化、数据增强与 Dropout。为了 CPU 冷启动，这里把通道数显著缩小，并用 32×32 灰度输入；层次关系仍由五个卷积与三层分类头表达。

`AdaptiveAvgPool2d` 固定分类头接口，避免把全连接输入维度和某个中间尺寸暗中绑定。它解决的是 shape 接口，不意味着任意分辨率都与训练分布等价。


In [ ]:
class AlexNetMini(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if in_channels <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("in_channels 必须为正")  # 遇到非法合同立即显式失败。
        self.in_channels = in_channels  # 计算并保存当前步骤的中间状态。
        self.features = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(in_channels, 16, 5, stride=2, padding=2), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
            nn.MaxPool2d(2),  # 执行当前语句以推进本节示例。
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(32, 48, 3, padding=1), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(48, 48, 3, padding=1), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(48, 32, 3, padding=1), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。
        self.pool = nn.AdaptiveAvgPool2d((2, 2))  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Flatten(), nn.Dropout(0.2), nn.Linear(32 * 2 * 2, 64),  # 执行当前语句以推进本节示例。
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, num_classes),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4 or x.shape[1] != self.in_channels:  # 按当前条件选择后续控制路径。
            raise ValueError(f"AlexNetMini 期望 [N,{self.in_channels},H,W]")  # 遇到非法合同立即显式失败。
        return self.classifier(self.pool(self.features(x)))  # 返回当前分支计算出的结果。

alex = AlexNetMini().to(DEVICE)  # 计算并保存当前步骤的中间状态。
alex.eval()  # 执行当前语句以推进本节示例。
alex_rgb = AlexNetMini(in_channels=3).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    alex_logits = alex(torch.zeros(2, 1, 32, 32))  # 计算并保存当前步骤的中间状态。
    alex_rgb_logits = alex_rgb(torch.zeros(2, 3, 32, 32))  # 计算并保存当前步骤的中间状态。
assert alex_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert alex_rgb_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert alex_rgb.features[0].in_channels == 3  # 用受控断言验证关键不变量。
assert any(isinstance(module, nn.Dropout) for module in alex.modules())  # 用受控断言验证关键不变量。


## 5. VGG：用重复的 3×3 block 管理复杂度

两个连续 3×3、stride=1 的卷积具有 5×5 有效感受野，同时插入两次非线性；相对单个 5×5 卷积通常更省参数。`VGGBlock` 把“若干 SAME 卷积 + 一次池化”封装为可组合单元，网络深度由 block 配置决定。

这里加入 BatchNorm 是现代化变体；原始 VGG 配置并非都带 BN。把变体写进类名和 manifest，避免把它误称为论文原模型。


In [ ]:
class VGGBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels, num_convs):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if num_convs < 1:  # 按当前条件选择后续控制路径。
            raise ValueError("每个 VGGBlock 至少包含一个卷积")  # 遇到非法合同立即显式失败。
        layers = []  # 计算并保存当前步骤的中间状态。
        for index in range(num_convs):  # 遍历输入元素以累积或检查结果。
            channels = in_channels if index == 0 else out_channels  # 计算并保存当前步骤的中间状态。
            layers += [nn.Conv2d(channels, out_channels, 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
                       nn.BatchNorm2d(out_channels), nn.ReLU()]  # 执行当前语句以推进本节示例。
        layers.append(nn.MaxPool2d(2))  # 执行当前语句以推进本节示例。
        self.block = nn.Sequential(*layers)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.block(x)  # 返回当前分支计算出的结果。

class VGGMini(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.features = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            VGGBlock(in_channels, 8, 2),  # 执行当前语句以推进本节示例。
            VGGBlock(8, 16, 2),  # 执行当前语句以推进本节示例。
            VGGBlock(16, 32, 2),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, num_classes))  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.head(self.features(x))  # 返回当前分支计算出的结果。

vgg = VGGMini().to(DEVICE)  # 计算并保存当前步骤的中间状态。
vgg_logits = vgg(torch.zeros(2, 1, 32, 32))  # 计算并保存当前步骤的中间状态。
assert vgg_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert sum(isinstance(m, nn.Conv2d) for m in vgg.modules()) == 6  # 用受控断言验证关键不变量。


## 6. ResNet：主分支学习残差，捷径必须满足 shape 契约

基本块计算 $y=\mathrm{ReLU}(F(x)+S(x))$。当输入输出通道和空间尺寸相同，$S(x)=x$；当 stride 或通道变化时，用 `1×1 Conv + BN` 投影。投影不是装饰：逐元素相加要求两个张量四个维度完全一致。

下面采用“两次 3×3 卷积、后激活”的基础块，接近原始 ResNet basic block；没有实现 bottleneck，也没有把论文中 ImageNet stem 原样搬到 32×32 小图。


In [ ]:
class BasicBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    expansion = 1  # 计算并保存当前步骤的中间状态。

    def __init__(self, in_channels, out_channels, stride=1):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride,  # 计算并保存当前步骤的中间状态。
                               padding=1, bias=False)  # 计算并保存当前步骤的中间状态。
        self.bn1 = nn.BatchNorm2d(out_channels)  # 计算并保存当前步骤的中间状态。
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)  # 计算并保存当前步骤的中间状态。
        self.bn2 = nn.BatchNorm2d(out_channels)  # 计算并保存当前步骤的中间状态。
        self.shortcut = (nn.Identity() if stride == 1 and in_channels == out_channels else  # 计算并保存当前步骤的中间状态。
                         nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, stride=stride,  # 计算并保存当前步骤的中间状态。
                                                bias=False), nn.BatchNorm2d(out_channels)))  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        identity = self.shortcut(x)  # 计算并保存当前步骤的中间状态。
        residual = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))  # 计算并保存当前步骤的中间状态。
        if residual.shape != identity.shape:  # 按当前条件选择后续控制路径。
            raise RuntimeError(f"残差 shape 不一致: {residual.shape} vs {identity.shape}")  # 遇到非法合同立即显式失败。
        return F.relu(residual + identity)  # 返回当前分支计算出的结果。

class MiniResNet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, num_classes=3, width=8):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.stem = nn.Sequential(nn.Conv2d(in_channels, width, 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
                                  nn.BatchNorm2d(width), nn.ReLU())  # 执行当前语句以推进本节示例。
        self.stage1 = nn.Sequential(BasicBlock(width, width), BasicBlock(width, width))  # 计算并保存当前步骤的中间状态。
        self.stage2 = nn.Sequential(BasicBlock(width, width * 2, stride=2),  # 计算并保存当前步骤的中间状态。
                                    BasicBlock(width * 2, width * 2))  # 执行当前语句以推进本节示例。
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),  # 计算并保存当前步骤的中间状态。
                                  nn.Linear(width * 2, num_classes))  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.head(self.stage2(self.stage1(self.stem(x))))  # 返回当前分支计算出的结果。

resnet = MiniResNet().to(DEVICE)  # 计算并保存当前步骤的中间状态。
resnet_logits = resnet(torch.zeros(2, 1, 32, 32))  # 计算并保存当前步骤的中间状态。
projection = resnet.stage2[0].shortcut  # 计算并保存当前步骤的中间状态。
assert resnet_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert isinstance(projection, nn.Sequential)  # 用受控断言验证关键不变量。
assert projection[0].kernel_size == (1, 1)  # 用受控断言验证关键不变量。


## 7. 初始化和参数量也是架构合同

ReLU 网络常用 Kaiming 初始化，使前向/反向方差不过快消失或爆炸；线性分类层同样可以使用。BatchNorm 的缩放初始化为 1、偏置为 0，使其初始接近标准化后的恒等仿射。初始化策略需要随激活函数变化，不能把 ReLU 的假设机械套到 sigmoid/tanh。

参数量只是容量指标，不等于 FLOPs、激活内存或延迟；生产评估仍需在目标硬件上以真实 batch 和输入尺寸基准测试。


In [ ]:
def init_cnn(module):  # 定义本节可复用的核心函数。
    if isinstance(module, nn.Conv2d):  # 按当前条件选择后续控制路径。
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")  # 计算并保存当前步骤的中间状态。
        if module.bias is not None:  # 按当前条件选择后续控制路径。
            nn.init.zeros_(module.bias)  # 执行当前语句以推进本节示例。
    elif isinstance(module, nn.Linear):  # 按当前条件选择后续控制路径。
        nn.init.kaiming_uniform_(module.weight, a=math.sqrt(5))  # 计算并保存当前步骤的中间状态。
        if module.bias is not None:  # 按当前条件选择后续控制路径。
            nn.init.zeros_(module.bias)  # 执行当前语句以推进本节示例。
    elif isinstance(module, nn.BatchNorm2d):  # 按当前条件选择后续控制路径。
        nn.init.ones_(module.weight)  # 执行当前语句以推进本节示例。
        nn.init.zeros_(module.bias)  # 执行当前语句以推进本节示例。

models = OrderedDict([  # 计算并保存当前步骤的中间状态。
    ("lenet", LeNet()), ("alexnet_mini", AlexNetMini()),  # 执行当前语句以推进本节示例。
    ("vgg_mini_bn", VGGMini()), ("mini_resnet", MiniResNet()),  # 执行当前语句以推进本节示例。
])  # 执行当前语句以推进本节示例。
for model in models.values():  # 遍历输入元素以累积或检查结果。
    model.apply(init_cnn)  # 执行当前语句以推进本节示例。
    model.to(DEVICE)  # 执行当前语句以推进本节示例。

parameter_counts = {name: sum(p.numel() for p in model.parameters() if p.requires_grad)  # 计算并保存当前步骤的中间状态。
                    for name, model in models.items()}  # 遍历输入元素以累积或检查结果。
assert all(count > 0 for count in parameter_counts.values())  # 用受控断言验证关键不变量。
assert len(set(parameter_counts.values())) == len(parameter_counts)  # 用受控断言验证关键不变量。
assert torch.isfinite(models["mini_resnet"].stem[0].weight).all()  # 用受控断言验证关键不变量。
print(parameter_counts)  # 执行当前语句以推进本节示例。


## 8. 用 forward hook 检查中间 shape

手算尺寸之后仍要让代码验证：hook 能记录实际执行到的叶子模块输入/输出，不改 `forward`。在复杂网络中可用它定位通道顺序、下采样次数和 flatten 维度错误。

hook 只适合诊断；忘记移除会持有引用并增加开销。下面在一次 dry-run 后立即 `remove()`。


In [ ]:
shape_trace = []  # 计算并保存当前步骤的中间状态。
handles = []  # 计算并保存当前步骤的中间状态。

def capture_shape(name):  # 定义本节可复用的核心函数。
    def hook(module, inputs, output):  # 定义本节可复用的核心函数。
        shape_trace.append((name, tuple(inputs[0].shape), tuple(output.shape)))  # 执行当前语句以推进本节示例。
    return hook  # 返回当前分支计算出的结果。

for name, module in models["mini_resnet"].named_modules():  # 遍历输入元素以累积或检查结果。
    if name and len(list(module.children())) == 0:  # 按当前条件选择后续控制路径。
        handles.append(module.register_forward_hook(capture_shape(name)))  # 执行当前语句以推进本节示例。

models["mini_resnet"].eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    dry_logits = models["mini_resnet"](torch.zeros(4, 1, 32, 32))  # 计算并保存当前步骤的中间状态。
for handle in handles:  # 遍历输入元素以累积或检查结果。
    handle.remove()  # 执行当前语句以推进本节示例。

assert dry_logits.shape == (4, 3)  # 用受控断言验证关键不变量。
assert shape_trace[0][1] == (4, 1, 32, 32)  # 用受控断言验证关键不变量。
assert any(out_shape[-2:] == (16, 16) for _, _, out_shape in shape_trace if len(out_shape) == 4)  # 用受控断言验证关键不变量。
assert len(handles) == len(shape_trace)  # 用受控断言验证关键不变量。
print("前 8 个叶子模块 shape:", shape_trace[:8])  # 执行当前语句以推进本节示例。


## 9. 失败反例：残差不能把不相容张量直接相加

主分支用 stride=2 且把 8 通道变为 16 通道后，shape 是 `[N,16,H/2,W/2]`；原始输入仍是 `[N,8,H,W]`。PyTorch 不会把这种结构性差异自动修好。捕获异常是为了把失败合同写成测试，而不是用 `try/except` 吞掉线上问题。


In [ ]:
x_bad = torch.randn(2, 8, 16, 16)  # 计算并保存当前步骤的中间状态。
main_bad = nn.Conv2d(8, 16, 3, stride=2, padding=1)(x_bad)  # 计算并保存当前步骤的中间状态。
residual_error = None  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    _ = main_bad + x_bad  # 计算并保存当前步骤的中间状态。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    residual_error = str(exc)  # 计算并保存当前步骤的中间状态。

fixed_block = BasicBlock(8, 16, stride=2).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    fixed = fixed_block(x_bad)  # 计算并保存当前步骤的中间状态。

assert residual_error is not None  # 用受控断言验证关键不变量。
assert main_bad.shape == (2, 16, 8, 8)  # 用受控断言验证关键不变量。
assert fixed.shape == (2, 16, 8, 8)  # 用受控断言验证关键不变量。
assert not isinstance(fixed_block.shortcut, nn.Identity)  # 用受控断言验证关键不变量。
print("预期失败:", residual_error.split("\n")[0])  # 执行当前语句以推进本节示例。


## 10. 可控合成数据：只验证学习链路

三类图像分别含竖条、横条和中心方块，并叠加轻微噪声与有限位置抖动。标签由生成机制决定，所以可以检查网络是否能记住/学习这个简单任务。训练、验证使用独立随机流，避免完全复制像素。

合成数据没有自然图像纹理、域偏移、长尾、标注噪声或对抗背景。小样本过拟合成功只说明 forward、loss、optimizer 与梯度大致贯通；失败则是强烈的实现告警。


In [ ]:
def make_patterns(n_per_class, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    images, labels = [], []  # 计算并保存当前步骤的中间状态。
    for label in range(3):  # 遍历输入元素以累积或检查结果。
        for _ in range(n_per_class):  # 遍历输入元素以累积或检查结果。
            image = torch.zeros(1, 32, 32)  # 计算并保存当前步骤的中间状态。
            shift = int(torch.randint(-2, 3, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
            if label == 0:  # 按当前条件选择后续控制路径。
                column = 16 + shift  # 计算并保存当前步骤的中间状态。
                image[:, 5:27, column-2:column+2] = 1.0  # 计算并保存当前步骤的中间状态。
            elif label == 1:  # 按当前条件选择后续控制路径。
                row = 16 + shift  # 计算并保存当前步骤的中间状态。
                image[:, row-2:row+2, 5:27] = 1.0  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                center = 16 + shift  # 计算并保存当前步骤的中间状态。
                image[:, center-5:center+5, center-5:center+5] = 1.0  # 计算并保存当前步骤的中间状态。
            noise = 0.06 * torch.randn(image.shape, generator=generator)  # 计算并保存当前步骤的中间状态。
            images.append((image + noise).clamp(0, 1))  # 执行当前语句以推进本节示例。
            labels.append(label)  # 执行当前语句以推进本节示例。
    return torch.stack(images), torch.tensor(labels, dtype=torch.long)  # 返回当前分支计算出的结果。

train_x, train_y = make_patterns(8, SEED + 1)  # 计算并保存当前步骤的中间状态。
val_x, val_y = make_patterns(4, SEED + 2)  # 计算并保存当前步骤的中间状态。
assert train_x.shape == (24, 1, 32, 32)  # 用受控断言验证关键不变量。
assert train_y.bincount().tolist() == [8, 8, 8]  # 用受控断言验证关键不变量。
assert not torch.equal(train_x[:4], val_x[:4])  # 用受控断言验证关键不变量。
assert 0.0 <= float(train_x.min()) <= float(train_x.max()) <= 1.0  # 用受控断言验证关键不变量。


## 11. 小样本过拟合测试

分类 logits 的 shape 是 `[N,C]`，`CrossEntropyLoss` 内部完成 `log_softmax + NLL`，所以 forward **不应先做 softmax**。优化器只接收当前模型参数，训练前后记录 loss 与准确率；验证集仅用于观察，不用于调出测试结论。

这里刻意让模型反复看到 24 个样本。这是单元测试式过拟合，不是合理的泛化训练制度。


In [ ]:
model22 = models["mini_resnet"]  # 计算并保存当前步骤的中间状态。
model22.train()  # 执行当前语句以推进本节示例。
optimizer22 = torch.optim.Adam(model22.parameters(), lr=0.02)  # 计算并保存当前步骤的中间状态。
criterion22 = nn.CrossEntropyLoss()  # 计算并保存当前步骤的中间状态。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss22 = float(criterion22(model22(train_x), train_y))  # 计算并保存当前步骤的中间状态。

loss_curve22 = []  # 计算并保存当前步骤的中间状态。
for step in range(70):  # 遍历输入元素以累积或检查结果。
    optimizer22.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model22(train_x)  # 计算并保存当前步骤的中间状态。
    loss = criterion22(logits, train_y)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    optimizer22.step()  # 执行当前语句以推进本节示例。
    loss_curve22.append(float(loss.detach()))  # 执行当前语句以推进本节示例。

model22.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    train_logits22 = model22(train_x)  # 计算并保存当前步骤的中间状态。
    val_logits22 = model22(val_x)  # 计算并保存当前步骤的中间状态。
    final_loss22 = float(criterion22(train_logits22, train_y))  # 计算并保存当前步骤的中间状态。
    train_acc22 = float((train_logits22.argmax(1) == train_y).float().mean())  # 计算并保存当前步骤的中间状态。
    val_acc22 = float((val_logits22.argmax(1) == val_y).float().mean())  # 计算并保存当前步骤的中间状态。

assert final_loss22 < initial_loss22 * 0.20  # 用受控断言验证关键不变量。
assert train_acc22 >= 0.95  # 用受控断言验证关键不变量。
assert len(loss_curve22) == 70  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in loss_curve22)  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss22, 4), "final_loss": round(final_loss22, 4),  # 执行当前语句以推进本节示例。
       "train_acc": train_acc22, "validation_acc": val_acc22})  # 执行当前语句以推进本节示例。


## 12. BatchNorm 的 train/eval 不是性能开关

训练态用当前 mini-batch 统计量并更新 running statistics；推理态使用已经积累的 running mean/variance。若服务忘记 `eval()`，同一样本可能随同批其他样本变化，running stats 还会继续漂移。反之，训练时误用 `eval()` 又会冻结统计量。

小 batch 下 BN 估计尤其不稳定，可能要比较 GroupNorm/LayerNorm、冻结 BN 或同步 BN。下面用独立模块展示状态变化，避免污染训练好的模型。


In [ ]:
bn_demo = nn.BatchNorm2d(2, momentum=0.5)  # 计算并保存当前步骤的中间状态。
batch_demo = torch.stack([torch.zeros(2, 4, 4), torch.full((2, 4, 4), 4.0)])  # 计算并保存当前步骤的中间状态。
before_mean = bn_demo.running_mean.clone()  # 计算并保存当前步骤的中间状态。
bn_demo.train()  # 执行当前语句以推进本节示例。
train_out = bn_demo(batch_demo)  # 计算并保存当前步骤的中间状态。
after_train_mean = bn_demo.running_mean.clone()  # 计算并保存当前步骤的中间状态。
bn_demo.eval()  # 执行当前语句以推进本节示例。
after_switch_mean = bn_demo.running_mean.clone()  # 计算并保存当前步骤的中间状态。
eval_out = bn_demo(batch_demo)  # 计算并保存当前步骤的中间状态。
after_eval_mean = bn_demo.running_mean.clone()  # 计算并保存当前步骤的中间状态。

assert torch.equal(before_mean, torch.zeros_like(before_mean))  # 用受控断言验证关键不变量。
assert not torch.equal(after_train_mean, before_mean)  # 用受控断言验证关键不变量。
assert torch.equal(after_switch_mean, after_eval_mean)  # 用受控断言验证关键不变量。
assert not torch.allclose(train_out, eval_out)  # 用受控断言验证关键不变量。
assert torch.isfinite(eval_out).all()  # 用受控断言验证关键不变量。
print({"before": before_mean.tolist(), "after_train": after_train_mean.tolist()})  # 执行当前语句以推进本节示例。


## 13. 梯度健康与可复现状态指纹

一次成功的 `backward()` 不保证每层得到有效信号。这里对待发布模型做 `deepcopy`，只在副本上执行反传，检查所有存在梯度的参数都是有限值，且至少一个卷积和分类头梯度非零。这样训练态探针不会改写原模型的 BatchNorm running statistics。

发布时不能只记录类名：还需保存架构配置、预处理、类别表、框架版本、随机性设置与权重摘要。下面的 SHA-256 对排序后的 `state_dict` 名称、dtype、shape 和原始字节做哈希；它适合一致性核验，不替代签名、制品库权限和安全序列化策略。


In [ ]:
# 梯度探针在深拷贝上运行，绝不改写待发布模型的参数或 BN buffer。
model22.eval()  # 执行当前语句以推进本节示例。
release_state_before22 = {name: tensor.detach().clone()  # 计算并保存当前步骤的中间状态。
                          for name, tensor in model22.state_dict().items()}  # 遍历输入元素以累积或检查结果。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    release_logits_before22 = model22(val_x).clone()  # 计算并保存当前步骤的中间状态。
probe_model22 = deepcopy(model22).train()  # 计算并保存当前步骤的中间状态。
probe_model22.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
probe_loss22 = criterion22(probe_model22(train_x[:6]), train_y[:6])  # 计算并保存当前步骤的中间状态。
probe_loss22.backward()  # 执行当前语句以推进本节示例。
gradient_rows22 = {name: float(parameter.grad.norm()) for name, parameter in probe_model22.named_parameters()  # 计算并保存当前步骤的中间状态。
                   if parameter.grad is not None}  # 按当前条件选择后续控制路径。

def state_dict_fingerprint(model):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(model.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        cpu_tensor = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(name.encode("utf-8"))  # 执行当前语句以推进本节示例。
        digest.update(str(cpu_tensor.dtype).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(str(tuple(cpu_tensor.shape)).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(cpu_tensor.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    release_logits_after22 = model22(val_x)  # 计算并保存当前步骤的中间状态。
release_state_unchanged22 = all(  # 计算并保存当前步骤的中间状态。
    torch.equal(tensor, release_state_before22[name])  # 执行当前语句以推进本节示例。
    for name, tensor in model22.state_dict().items()  # 遍历输入元素以累积或检查结果。
)  # 执行当前语句以推进本节示例。
fingerprint22 = state_dict_fingerprint(model22)  # 计算并保存当前步骤的中间状态。
manifest22 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "mini-resnet-width8-toy-v1",  # 执行当前语句以推进本节示例。
    "architecture": {"block": "BasicBlock", "width": 8, "classes": 3},  # 执行当前语句以推进本节示例。
    "input_contract": {"layout": "NCHW", "shape": [None, 1, 32, 32], "range": [0.0, 1.0]},  # 执行当前语句以推进本节示例。
    "optimizer": {"name": "Adam", "lr": 0.02, "overfit_steps": 70},  # 执行当前语句以推进本节示例。
    "seed": SEED,  # 执行当前语句以推进本节示例。
    "torch": torch.__version__,  # 执行当前语句以推进本节示例。
    "state_dict_sha256": fingerprint22,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

assert gradient_rows22  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in gradient_rows22.values())  # 用受控断言验证关键不变量。
assert gradient_rows22["stem.0.weight"] > 0  # 用受控断言验证关键不变量。
assert gradient_rows22["head.2.weight"] > 0  # 用受控断言验证关键不变量。
assert release_state_unchanged22  # 用受控断言验证关键不变量。
assert torch.equal(release_logits_before22, release_logits_after22)  # 用受控断言验证关键不变量。
assert torch.equal(release_logits_before22, val_logits22)  # 用受控断言验证关键不变量。
assert model22.training is False  # 用受控断言验证关键不变量。
assert len(fingerprint22) == 64  # 用受控断言验证关键不变量。
assert state_dict_fingerprint(model22) == fingerprint22  # 用受控断言验证关键不变量。
print(json.dumps(manifest22, ensure_ascii=False, indent=2))  # 计算并保存当前步骤的中间状态。


## 14. 生产边界、排障顺序与资料

真实系统还需要：按来源/时间切分；只在训练集拟合归一化和增强；混合精度数值回归；显存峰值和 p50/p95/p99 延迟；输入解码与尺寸上限；类别映射版本；校准、拒识、漂移和分群指标；导出到 TorchScript/ONNX 后与 eager 的逐样本容差测试；可回滚制品与审计日志。

排障建议从合同开始：① 单样本 shape/range；② 每层 hook；③ logits 是否未经 softmax 就进入 CE；④ train/eval；⑤ 梯度有限且非零；⑥ 小样本能否过拟合；⑦ 再看增强、学习率和真实数据问题。

原始论文与官方资料：

- LeCun 等，[Gradient-Based Learning Applied to Document Recognition](https://doi.org/10.1109/5.726791)，1998。
- Krizhevsky 等，[ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html)，2012。
- Simonyan 与 Zisserman，[Very Deep Convolutional Networks for Large-Scale Image Recognition](https://arxiv.org/abs/1409.1556)，2014。
- He 等，[Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)，2015。
- PyTorch 官方文档：[Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)、[BatchNorm2d](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)。

结论边界：本 notebook 证明这些手写类在受控 CPU 小数据上满足 shape、梯度和过拟合测试；它不证明论文级复现精度，也不证明真实业务泛化。
